In [1]:
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import os

### **Config**

In [2]:
MODEL_ID    = "Salesforce/codegen-350M-mono"
MAX_LEN     = 512
TARGET_LANG = {1}          # 1=Python  2=C++  3=Java
LANG_MAP    = {1: "Python", 2: "C++", 3: "Java"}

DRIVE_ROOT  = "/content/drive/MyDrive/codegen_cp"
TOK_DIR     = f"{DRIVE_ROOT}/tokenizer"
DATA_PATH   = f"{DRIVE_ROOT}/processed/tokenized_data.pt"

### **Mount Drive**

In [3]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/processed", exist_ok=True)

Mounted at /content/drive


In [4]:
def load_tokenizer(model_id: str, save_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token_id is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        print("[PAD] token added")
    else:
        print("Tokenizer already has a pad token")
    tokenizer.save_pretrained(save_dir)
    print(f"Tokenizer saved → {save_dir}")
    return tokenizer

tokenizer = load_tokenizer(MODEL_ID, TOK_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

[PAD] token added
Tokenizer saved → /content/drive/MyDrive/codegen_cp/tokenizer


In [5]:
data = load_dataset("deepmind/code_contests", split="train")
print(f"Dataset loaded: {len(data)} examples")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00039-e991a271dbfa99(…):   0%|          | 0.00/180M [00:00<?, ?B/s]

data/train-00001-of-00039-e092fe56fda187(…):   0%|          | 0.00/209M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/13328 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/165 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/117 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

Dataset loaded: 13328 examples


In [6]:
# ── 6. Flatten to (problem, language, solution) rows ─────────
def flatten_problems(example):
    rows = []
    languages  = example["solutions"]["language"]
    solutions  = example["solutions"]["solution"]
    for sol, lang in zip(solutions, languages):
        if lang not in TARGET_LANG:
            continue
        rows.append({
            "problem" : example["description"],
            "language": LANG_MAP.get(lang, "unk"),
            "solution": sol,          # ← plain string, NOT a list
        })
    return rows

flat_data = []
for example in tqdm(data, desc="Flattening"):
    flat_data.extend(flatten_problems(example))
print(f"Flat rows: {len(flat_data)}")

Flattening: 100%|██████████| 13328/13328 [00:18<00:00, 703.80it/s]

Flat rows: 290300


In [7]:
# ── 7. Tokenise ──────────────────────────────────────────────
def pre_process(example):
    # NOTE: example['solution'] is already a string here
    text = (
        f"You are an expert competitive programmer.\n"
        f"Solve the following problem in {example['language']}.\n\n"
        f"Problem:\n{example['problem']}\n\n"
        f"Solution:\n{example['solution']}"
    )

    tokens = tokenizer(
        text,
        max_length=MAX_LEN - 1,   # leave room for EOS
        truncation=True,
        padding=False,
    )

    # Append EOS
    tokens["input_ids"].append(tokenizer.eos_token_id)
    tokens["attention_mask"].append(1)

    # Right-pad to MAX_LEN
    pad_len = MAX_LEN - len(tokens["input_ids"])
    tokens["input_ids"]       += [tokenizer.pad_token_id] * pad_len
    tokens["attention_mask"]  += [0] * pad_len

    # Labels: mask padding
    tokens["labels"] = [
        -100 if t == tokenizer.pad_token_id else t
        for t in tokens["input_ids"]
    ]

    return {
        "input_ids":      tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels":         tokens["labels"],
    }

In [8]:
tokenized_data = []
skipped = 0
for example in tqdm(flat_data, desc="Tokenising"):
    try:
        tokenized_data.append(pre_process(example))
    except Exception as e:
        skipped += 1

print(f"Tokenised: {len(tokenized_data)}  |  Skipped: {skipped}")

Tokenising: 100%|██████████| 290300/290300 [08:36<00:00, 562.60it/s]

✅ Tokenised: 290300  |  Skipped: 0


In [9]:
# ── 8. Save to Drive ─────────────────────────────────────────
torch.save(tokenized_data, DATA_PATH)
print(f"Tokenized data saved → {DATA_PATH}")

# Quick sanity check
sample = tokenized_data[0]
assert len(sample["input_ids"])      == MAX_LEN
assert len(sample["attention_mask"]) == MAX_LEN
assert len(sample["labels"])         == MAX_LEN

Tokenized data saved → /content/drive/MyDrive/codegen_cp/processed/tokenized_data.pt
